# 02 - Utility experiments

This notebook tests whether the EEG keeps useful task information. The task is left fist vs right fist motor imagery, and the label is `y`.

## Contents

1. Setup and data loading
2. EEGNet helper functions
3. Random trial split
4. Run-held-out split
5. Save results

## 1. Setup

Mount Drive, install dependencies, import packages, and set a seed.

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Drive mount skipped. This is fine outside Colab.")

Mounted at /content/drive


In [3]:
!pip -q install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 104.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [5]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

from tensorflow.keras import layers, models, constraints
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed()
print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


## 2. Load the run-aware dataset

This notebook uses the same `with_runs` file as the privacy notebook.

In [6]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_PATH = DRIVE_ROOT / "URV_Datasets" / "physionet_mi_lr_imagery_subjects_1_50_with_runs.npz"
RESULTS_DIR = DRIVE_ROOT / "URV_Datasets" / "results"
RESULTS_CSV = RESULTS_DIR / "physionet_utility_results_clean.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_PATH)
print("Results:", RESULTS_CSV)

Data: /content/drive/MyDrive/URV_Datasets/physionet_mi_lr_imagery_subjects_1_50_with_runs.npz
Results: /content/drive/MyDrive/URV_Datasets/results/physionet_utility_results_clean.csv


In [7]:
def load_dataset(path):
    data = np.load(path, allow_pickle=False)
    required = ["X", "y", "subject_ids", "run_ids"]
    missing = [key for key in required if key not in data.files]
    if missing:
        raise KeyError(f"Missing keys: {missing}")

    X = data["X"].astype("float32")
    y = data["y"].astype("int64")
    subject_ids = data["subject_ids"].astype("int64")
    run_ids = data["run_ids"].astype("int64")

    if X.ndim != 3:
        raise ValueError(f"Expected X as trials x channels x samples, got {X.shape}")
    if not (len(X) == len(y) == len(subject_ids) == len(run_ids)):
        raise ValueError("X, y, subject_ids, and run_ids do not have the same length.")

    return X, y, subject_ids, run_ids


X, y, subject_ids, run_ids = load_dataset(DATA_PATH)

print("X:", X.shape)
print("y:", y.shape)
print("subject_ids:", subject_ids.shape)
print("run_ids:", run_ids.shape)
print("Task labels:", dict(zip(*np.unique(y, return_counts=True))))
print("Runs:", dict(zip(*np.unique(run_ids, return_counts=True))))
print("Subjects:", len(np.unique(subject_ids)))

X: (2245, 64, 641)
y: (2245,)
subject_ids: (2245,)
run_ids: (2245,)
Task labels: {np.int64(0): np.int64(1132), np.int64(1): np.int64(1113)}
Runs: {np.int64(4): np.int64(750), np.int64(8): np.int64(747), np.int64(12): np.int64(748)}
Subjects: 50


## 3. EEGNet helpers

These cells handle per-trial normalization, EEGNet input shape, model construction, and evaluation.

In [8]:
def zscore_per_trial_channel(X, eps=1e-6):
    mean = X.mean(axis=2, keepdims=True)
    std = X.std(axis=2, keepdims=True)
    return ((X - mean) / (std + eps)).astype("float32")


def prepare_eegnet_inputs(X_train, X_test, y_train, y_test):
    X_train_eegnet = X_train[..., np.newaxis]
    X_test_eegnet = X_test[..., np.newaxis]

    classes = np.unique(y_train)
    class_to_index = {label: idx for idx, label in enumerate(classes)}
    y_train_enc = np.array([class_to_index[label] for label in y_train])
    y_test_enc = np.array([class_to_index[label] for label in y_test])

    y_train_cat = to_categorical(y_train_enc, num_classes=len(classes))
    y_test_cat = to_categorical(y_test_enc, num_classes=len(classes))

    return X_train_eegnet, X_test_eegnet, y_train_enc, y_test_enc, y_train_cat, y_test_cat

In [9]:
def build_eegnet_binary(n_channels, n_samples, n_classes=2, dropout_rate=0.5):
    inputs = layers.Input(shape=(n_channels, n_samples, 1))

    x = layers.Conv2D(8, (1, 64), padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)

    x = layers.DepthwiseConv2D(
        (n_channels, 1),
        use_bias=False,
        depth_multiplier=2,
        depthwise_constraint=constraints.max_norm(1.0),
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.SeparableConv2D(16, (1, 16), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(
        n_classes,
        activation="softmax",
        kernel_constraint=constraints.max_norm(0.25),
    )(x)

    return models.Model(inputs, outputs)

In [10]:
def train_eegnet(X_train_raw, X_test_raw, y_train_raw, y_test_raw):
    set_seed()
    tf.keras.backend.clear_session()

    # Normalize train and test trials independently.
    X_train_norm = zscore_per_trial_channel(X_train_raw)
    X_test_norm = zscore_per_trial_channel(X_test_raw)

    (
        X_train_eegnet,
        X_test_eegnet,
        y_train_enc,
        y_test_enc,
        y_train_cat,
        y_test_cat,
    ) = prepare_eegnet_inputs(X_train_norm, X_test_norm, y_train_raw, y_test_raw)

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train_eegnet,
        y_train_cat,
        test_size=0.20,
        random_state=SEED,
        stratify=y_train_enc,
    )

    model = build_eegnet_binary(
        n_channels=X_train_eegnet.shape[1],
        n_samples=X_train_eegnet.shape[2],
        n_classes=y_train_cat.shape[1],
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5),
    ]

    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(X_val, y_val),
        epochs=150,
        batch_size=32,
        callbacks=callbacks,
        verbose=1,
    )

    test_loss, keras_acc = model.evaluate(X_test_eegnet, y_test_cat, verbose=0)
    y_pred = np.argmax(model.predict(X_test_eegnet, verbose=0), axis=1)

    metrics = {
        "accuracy": accuracy_score(y_test_enc, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test_enc, y_pred),
        "f1_macro": f1_score(y_test_enc, y_pred, average="macro"),
        "f1_weighted": f1_score(y_test_enc, y_pred, average="weighted"),
        "keras_test_loss": float(test_loss),
        "keras_test_accuracy": float(keras_acc),
        "confusion_matrix": confusion_matrix(y_test_enc, y_pred),
        "report": classification_report(
            y_test_enc,
            y_pred,
            target_names=["Left fist", "Right fist"],
        ),
    }

    return model, history, metrics

In [11]:
results = []


def add_result(name, split, metrics, n_train, n_test, train_runs, test_runs):
    results.append({
        "experiment": name,
        "dataset": "PhysioNet Motor Imagery",
        "subjects": "1-50",
        "target": "left/right motor imagery",
        "label": "y",
        "split": split,
        "model": "EEGNet",
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
        "keras_test_loss": metrics["keras_test_loss"],
        "keras_test_accuracy": metrics["keras_test_accuracy"],
        "n_train": n_train,
        "n_test": n_test,
        "train_runs": train_runs,
        "test_runs": test_runs,
        "normalization": "per-trial per-channel z-score",
    })

## Experiment 1: random trial split

This is the easier utility check. Trials from the same subject and run can appear on both sides of the split.

In [12]:
X_train, X_test, y_train, y_test, sub_train, sub_test, run_train, run_test = train_test_split(
    X,
    y,
    subject_ids,
    run_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print("Train:", X_train.shape, dict(zip(*np.unique(y_train, return_counts=True))))
print("Test:", X_test.shape, dict(zip(*np.unique(y_test, return_counts=True))))

random_model, random_history, random_metrics = train_eegnet(X_train, X_test, y_train, y_test)

print("Accuracy:", random_metrics["accuracy"])
print("Balanced accuracy:", random_metrics["balanced_accuracy"])
print("F1 weighted:", random_metrics["f1_weighted"])
print(random_metrics["report"])
print(random_metrics["confusion_matrix"])

add_result(
    name="Utility Exp 1",
    split="random stratified trial split",
    metrics=random_metrics,
    n_train=len(y_train),
    n_test=len(y_test),
    train_runs="mixed",
    test_runs="mixed",
)

Train: (1796, 64, 641) {np.int64(0): np.int64(906), np.int64(1): np.int64(890)}
Test: (449, 64, 641) {np.int64(0): np.int64(226), np.int64(1): np.int64(223)}
Epoch 1/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 20s 231ms/step - accuracy: 0.5028 - loss: 0.6977 - val_accuracy: 0.5083 - val_loss: 0.6929 - learning_rate: 0.0010
Epoch 2/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5007 - loss: 0.6962 - val_accuracy: 0.5083 - val_loss: 0.6929 - learning_rate: 0.0010
Epoch 3/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5223 - loss: 0.6927 - val_accuracy: 0.5306 - val_loss: 0.6927 - learning_rate: 0.0010
Epoch 4/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5292 - loss: 0.6895 - val_accuracy: 0.5639 - val_loss: 0.6921 - learning_rate: 0.0010
Epoch 5/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5439 - loss: 0.6879 - val_accuracy: 0.5528 - val_loss: 0.6911 - learning_rate: 0.0010
Epoch 6/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5543 - loss: 0

## 4. Random trial split

This is the easier utility setting. It is useful as a sanity check, but trials from the same subject and run can appear in both train and test sets.

## Experiment 2: run-held-out split

Train on runs 4 and 8. Test on run 12.

## 5. Run-held-out split

Train on runs 4 and 8, then test on run 12. This is the stronger utility result because the test set comes from a different recording run.

In [13]:
train_mask = np.isin(run_ids, [4, 8])
test_mask = run_ids == 12

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print("Train:", X_train.shape, dict(zip(*np.unique(y_train, return_counts=True))))
print("Test:", X_test.shape, dict(zip(*np.unique(y_test, return_counts=True))))
print("Train runs:", dict(zip(*np.unique(run_ids[train_mask], return_counts=True))))
print("Test runs:", dict(zip(*np.unique(run_ids[test_mask], return_counts=True))))
print("Train subjects:", len(np.unique(subject_ids[train_mask])))
print("Test subjects:", len(np.unique(subject_ids[test_mask])))

run_model, run_history, run_metrics = train_eegnet(X_train, X_test, y_train, y_test)

print("Accuracy:", run_metrics["accuracy"])
print("Balanced accuracy:", run_metrics["balanced_accuracy"])
print("F1 weighted:", run_metrics["f1_weighted"])
print(run_metrics["report"])
print(run_metrics["confusion_matrix"])

add_result(
    name="Utility Exp 2",
    split="run-held-out: train runs 4 and 8, test run 12",
    metrics=run_metrics,
    n_train=len(y_train),
    n_test=len(y_test),
    train_runs="4,8",
    test_runs="12",
)

Train: (1497, 64, 641) {np.int64(0): np.int64(756), np.int64(1): np.int64(741)}
Test: (748, 64, 641) {np.int64(0): np.int64(376), np.int64(1): np.int64(372)}
Train runs: {np.int64(4): np.int64(750), np.int64(8): np.int64(747)}
Test runs: {np.int64(12): np.int64(748)}
Train subjects: 50
Test subjects: 50
Epoch 1/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 15s 213ms/step - accuracy: 0.4787 - loss: 0.7014 - val_accuracy: 0.4900 - val_loss: 0.6932 - learning_rate: 0.0010
Epoch 2/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5013 - loss: 0.6953 - val_accuracy: 0.4967 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 3/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5221 - loss: 0.6938 - val_accuracy: 0.5000 - val_loss: 0.6932 - learning_rate: 0.0010
Epoch 4/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5347 - loss: 0.6903 - val_accuracy: 0.4833 - val_loss: 0.6931 - learning_rate: 0.0010
Epoch 5/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.5489 - loss: 0.6865

## 6. Save results

Write one CSV row per experiment.

In [14]:
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
display(results_df)
print("Saved:", RESULTS_CSV)

,experiment,dataset,subjects,target,label,split,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,keras_test_loss,keras_test_accuracy,n_train,n_test,train_runs,test_runs,normalization
0,Utility Exp 1,PhysioNet Motor Imagery,1-50,left/right motor imagery,y,random stratified trial split,EEGNet,0.788419,0.788305,0.788314,0.788345,0.456790,0.788419,1796,449,mixed,mixed,per-trial per-channel z-score
1,Utility Exp 2,PhysioNet Motor Imagery,1-50,left/right motor imagery,y,"run-held-out: train runs 4 and 8, test run 12",EEGNet,0.720588,0.720387,0.720107,0.720170,0.549625,0.720588,1497,748,"4,8",12,per-trial per-channel z-score


Saved: /content/drive/MyDrive/URV_Datasets/results/physionet_utility_results_clean.csv
